<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/prompt_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

Cloning into 'Dungeons-and-Dragons-Turn-Classification'...
remote: Enumerating objects: 416, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 416 (delta 126), reused 41 (delta 41), pack-reused 233 (from 1)
Receiving objects: 100% (416/416), 3.05 MiB | 10.96 MiB/s, done.
Resolving deltas: 100% (243/243), done.


In [ ]:
!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 27.0 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.2
    Uninstalling typeguard-4.5.2:
      Successfully uninstalled typeguard-4.5.2


In [ ]:
import pandas as pd
import dspy

In [ ]:
training_df = pd.read_excel('Dungeons-and-Dragons-Turn-Classification/pipeline_testing_set.xlsx')

In [ ]:
training_df

,Index,context,current turn,category,Unnamed: 4
0,433,Dialogue context: MATT: This is a survival. LA...,LAURA: That's where the acid pits are. Acid. B...,Worldbuilding,NaN
1,435,"Dialogue context: MATT: 16, okay. You get down...","MATT: All right. Continuing to push forward, a...",Worldbuilding,NaN
2,436,Dialogue context: LAURA: That's where the acid...,LIAM: Looks a little hinky?,Worldbuilding,NaN
3,437,Dialogue context: TRAVIS: Yeah. MATT: All righ...,"MATT: It looks like it's deliberately placed, ...",Worldbuilding,NaN
4,438,Dialogue context: MATT: All right. Continuing ...,LIAM: I'd like to carefully approach and disar...,Strategising,NaN
...,...,...,...,...,...
395,857,Dialogue context: MATT: Okay. The bullet slams...,TRAVIS: What about these jewels?,Worldbuilding,NaN
396,858,Dialogue context: LIAM: I have advantage. MATT...,MATT: The Feast gives you advantage.,Gameplay Mechanic,NaN
397,859,Dialogue context: MATT: You do. TRAVIS: What a...,"LIAM: The Feast gives me advantage, well, but ...",Gameplay Mechanic,NaN
398,860,Dialogue context: TRAVIS: What about these jew...,MATT: You rolled a 16? All right.,Gameplay Mechanic,NaN


In [ ]:
training_df.drop(columns=['Index'], inplace=True)

In [ ]:
training_df.drop(columns=['Unnamed: 4'], inplace=True)

In [ ]:
training_df

,context,current turn,category
0,Dialogue context: MATT: This is a survival. LA...,LAURA: That's where the acid pits are. Acid. B...,Worldbuilding
1,"Dialogue context: MATT: 16, okay. You get down...","MATT: All right. Continuing to push forward, a...",Worldbuilding
2,Dialogue context: LAURA: That's where the acid...,LIAM: Looks a little hinky?,Worldbuilding
3,Dialogue context: TRAVIS: Yeah. MATT: All righ...,"MATT: It looks like it's deliberately placed, ...",Worldbuilding
4,Dialogue context: MATT: All right. Continuing ...,LIAM: I'd like to carefully approach and disar...,Strategising
...,...,...,...
395,Dialogue context: MATT: Okay. The bullet slams...,TRAVIS: What about these jewels?,Worldbuilding
396,Dialogue context: LIAM: I have advantage. MATT...,MATT: The Feast gives you advantage.,Gameplay Mechanic
397,Dialogue context: MATT: You do. TRAVIS: What a...,"LIAM: The Feast gives me advantage, well, but ...",Gameplay Mechanic
398,Dialogue context: TRAVIS: What about these jew...,MATT: You rolled a 16? All right.,Gameplay Mechanic


In [ ]:
trainset = []

for context, current_turn, category in training_df.values:
    examples = dspy.Example(context=context, question=current_turn, response=category).with_inputs("context", "question")
    trainset.append(examples)

In [ ]:
trainset[:5]

[Example({'context': "Dialogue context: MATT: This is a survival. LAURA: Oh. Well, it's the same. That is a 16. MATT: 16, okay. You get down and inspect the area. You actually pull out a small bit of torch pith and light it a bit just to get a little light across this one intersection briefly. There are footprints back and forth in all ways, but many of them have been, at least the footprints coming from, and the last footprints were coming from this tunnel to your immediate left, were from a while back. The fresher footprints, fresh as of four hours, continue on down that way. You do notice that the smell of the air itself is, you notice this as well as you're inspecting, the air down here has a very acrid, sharp scent anyway, a hint of chemical. It seems to be getting stronger to the left.", 'question': "LAURA: That's where the acid pits are. Acid. Briarwoods. Briarwoods?", 'response': 'Worldbuilding'}) (input_keys={'context', 'question'}),
 Example({'context': "Dialogue context: MAT

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=2048)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal


class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['Gameplay Mechanic',
                      'Out-Of-Game Conversation',
                      'Worldbuilding',
                      'Strategising',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  category: Literal['Gameplay Mechanic',
                    'Out-Of-Game Conversation',
                    'Worldbuilding',
                    'Strategising',
                    ] = dspy.InputField()
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game.")

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.ChainOfThought(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction


In [ ]:
classify = ClassifyTurns()
classify.load("optimized_classifier.json")

In [ ]:
class PromptGenerator(dspy.Module):
  def __init__(self):
    self.classifier = classify
    self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

  def forward(self, context, question, **kwargs):
    pred_category = classify(context=context, question=question)
    prompt = self.generator(category=pred_category)
    return prompt, pred_category

prompt_generator = PromptGenerator()


In [ ]:
prompt_generator(trainset[1]['context'], trainset[1]['question'])

2026/06/15 15:29:58 WARNING dspy.predict.predict: Type mismatch for field 'category': expected Literal['Gameplay Mechanic', 'Out-Of-Game Conversation', 'Worldbuilding', 'Strategising'] based on given Signature, but the provided value is incompatible: Prediction(
    reasoning="Matt's description of the environment, including the placement of stone chunks, platforms, and the acrid scent, contributes to the game's setting and spatial awareness. This enhances the players' understanding of the world they're navigating, which is a key aspect of worldbuilding in a role-playing game.",
    response='Worldbuilding'
).


(Prediction(
     reasoning="Matt's description of the environment, including the placement of stone chunks, platforms, and the acrid scent, contributes to the game's setting and spatial awareness. This enhances the players' understanding of the world they're navigating, which is a key aspect of worldbuilding in a role-playing game.",
     player_instruction="Describe the environment in detail, including sensory elements (sights, sounds, smells) and spatial arrangements, to enrich the game world and guide your character's interactions. Use these details to inform your decisions and immerse your fellow players in the setting."
 ),
 Prediction(
     reasoning="Matt's description of the environment, including the placement of stone chunks, platforms, and the acrid scent, contributes to the game's setting and spatial awareness. This enhances the players' understanding of the world they're navigating, which is a key aspect of worldbuilding in a role-playing game.",
     response='Worldbuild